### **Методы предобработки текста**

1. Just Tokenization - Процесс разбиения текста на отдельные токены (обычно слова или подслова) без изменения их формы.

2. Stemming - Метод приведения слов к основе путём механического отсечения окончаний без учёта контекста и грамматики.

3. Lemmatization - Процесс приведения слова к его словарной (нормальной) форме с учётом морфологии и части речи.

4. Stemming + Misspellings - Комбинация исправления орфографических ошибок и последующим стреммиингом.

5. Lemmatization + Misspellings - Комбинация исправления орфографических ошибок и последующей лемматизации для получения корректной нормальной формы слов.

### **Методы векторизации текста**

1. Binary (0/1) - Бинарное представление текста, при котором каждому слову соответствует 1, если оно присутствует в документе, и 0 — если отсутствует.

2. Word Counts (Bag of Words) - Модель представления текста, в которой каждому слову сопоставляется количество его появлений в документе.

3. TF-IDF - Статистическая мера, отражающая важность слова в документе с учётом его частоты в данном документе и редкости во всём корпусе.

### 1. Подготовка данных

In [144]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler

from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag, word_tokenize
from nltk.corpus import wordnet

from tqdm import tqdm
from spellchecker import SpellChecker
from gensim.models import Word2Vec

import warnings
warnings.filterwarnings('ignore')

In [128]:
positive = pd.read_csv('data/processedPositive.csv', header=None).T.rename(columns={0: 'tweet_text'})
neutral = pd.read_csv('data/processedNeutral.csv', header=None).T.rename(columns={0: 'tweet_text'})
negative = pd.read_csv('data/processedNegative.csv', header=None).T.rename(columns={0: 'tweet_text'})
results = pd.DataFrame(columns=['Method', "Vectorizer", 'Model', 'Params', 'Accuracy'])

positive['label'] = 1
neutral['label'] = 0
negative['label'] = -1

df = pd.concat([positive, neutral, negative], ignore_index=True)
df['tweet_text'] = df['tweet_text'].astype('str')

X = df.drop(columns=['label'])
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=21
)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(3098, 1) (775, 1) (3098,) (775,)


In [252]:
X_train, X_test, y_train, y_test = X_train.reset_index(drop=True), X_test.reset_index(
    drop=True), y_train.reset_index(drop=True), y_test.reset_index(drop=True)

### 2. Реализация методов предобработки

##### 1. Just Tokenization

In [260]:
def vectorize(X_train, X_test, vectorizer_column):
    binary_vectorizer = CountVectorizer(binary=True)
    count_vectorizer = CountVectorizer()
    tfidf_vectorizer = TfidfVectorizer()

    X_train_binary = binary_vectorizer.fit_transform(X_train[vectorizer_column])
    X_test_binary = binary_vectorizer.transform(X_test[vectorizer_column])

    X_train_count = count_vectorizer.fit_transform(X_train[vectorizer_column])
    X_test_count = count_vectorizer.transform(X_test[vectorizer_column])

    X_train_tfidf = tfidf_vectorizer.fit_transform(X_train[vectorizer_column])
    X_test_tfidf = tfidf_vectorizer.transform(X_test[vectorizer_column])

    print("Binary shape:", X_train_binary.shape)
    print("Count shape:", X_train_count.shape)
    print("TF-IDF shape:", X_train_tfidf.shape)

    return X_train_binary, X_test_binary, X_train_count, X_test_count, X_train_tfidf, X_test_tfidf


X_train_binary_1, X_test_binary_1, X_train_count_1, X_test_count_1, X_train_tfidf_1, X_test_tfidf_1 = vectorize(X_train,
                                                                                                                X_test,
                                                                                                                'tweet_text')

Binary shape: (3098, 5673)
Count shape: (3098, 5673)
TF-IDF shape: (3098, 5673)


In [261]:
def search_top10(matrix):
    matrix = matrix.astype(float)

    cos_sim_matrix = cosine_similarity(matrix)
    np.fill_diagonal(cos_sim_matrix, 0)

    pairs = []
    for i in range(cos_sim_matrix.shape[0]):
        for j in range(i + 1, cos_sim_matrix.shape[1]):
            score = cos_sim_matrix[i, j]
            if score >= 1.0 - 1e-5:
                continue
            pairs.append((i, j, score))

    pairs_sorted = sorted(pairs, key=lambda x: x[2], reverse=True)

    # Берем 10 лучших
    top_10 = pairs_sorted[:10]

    # Выводим пары и сходство
    for idx1, idx2, score in top_10:
        print(f"Score: {score:.3f}")
        print(f"Tweet 1: {X_train.loc[idx1, 'tweet_text']}")
        print(f"Tweet 2: {X_train.loc[idx2, 'tweet_text']}")
        print('-' * 50)


def search_all_top10(X_train_binary, X_train_count, X_train_tfidf):
    print("Binary_vectorizer:")
    search_top10(X_train_binary)
    print("Count vectorizer:")
    search_top10(X_train_count)
    print("TF-IDF vectorizer:")
    search_top10(X_train_tfidf)


search_all_top10(X_train_binary_1, X_train_count_1, X_train_tfidf_1)

Binary_vectorizer:
Score: 0.978
Tweet 1: Hi! We tried to call your number but got no response unhappy  Please share another suitable time and an alternate number for us to.. cont1
Tweet 2: Hi! We tried to call your number but got no response unhappy  Please share another suitable time and an alternate number for.. cont1
--------------------------------------------------
Score: 0.977
Tweet 1: Hi! We tried to call your number but got no response unhappy  Please share another suitable time and an alternate number.. cont1
Tweet 2: Hi! We tried to call your number but got no response unhappy  Please share another suitable time and an alternate number for.. cont1
--------------------------------------------------
Score: 0.977
Tweet 1: Hi! We tried to call your number but got no response unhappy  Please share another suitable time and an alternate number.. cont1
Tweet 2: Hi Ashish! We tried to call your number but got no response unhappy  Please share another suitable time and an alternate.. 

In [262]:
def grid_search_train(model, param_grid, X_train, X_test, y_train, y_test, method, vectorize, model_name):
    grid = GridSearchCV(model, param_grid, cv=5, scoring='f1_macro', n_jobs=-1)
    grid.fit(X_train, y_train)

    y_pred = grid.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    print(f"Method: {method}, Model: {model_name}")
    print(f"Best params: {grid.best_params_}")
    print(f"Best accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred))
    print('-' * 50)

    # Перезапись или добавление в results
    global results
    mask = (results['Method'] == method) & (results['Model'] == model_name) & (results['Vectorizer'] == vectorize)
    if mask.any():
        results.loc[mask, 'Params'] = str(grid.best_params_)
        results.loc[mask, 'Accuracy'] = acc
    else:
        results = pd.concat([results, pd.DataFrame([{
            'Method': method,
            "Vectorizer": vectorize,
            'Model': model_name,
            'Params': str(grid.best_params_),
            'Accuracy': acc
        }])], ignore_index=True)

    return grid.best_estimator_


# Logistic Regression
param_grid_lr = {
    'C': [0.1, 1, 10],
    'penalty': ['l2'],
    'solver': ['lbfgs'],
    'max_iter': [100, 500]
}

# MultinomialNB
param_grid_nb = {
    'alpha': [0.1, 0.5, 1.0]
}


def run_all_model(X_train, X_test, y_train, y_test, vectorize, method):
    # Logistic Regression
    best_lr = grid_search_train(LogisticRegression(random_state=21), param_grid_lr,
                                X_train, X_test, y_train, y_test, method, vectorize, 'LogisticRegression')

    # MultinomialNB
    best_nb = grid_search_train(MultinomialNB(), param_grid_nb,
                                X_train, X_test, y_train, y_test, method, vectorize, 'MultinomialNB')

    return best_lr, best_nb


best_lr_1_1, best_nb_1_1 = run_all_model(X_train_binary_1, X_test_binary_1, y_train, y_test, 'Binary',
                                         'Just Tokenization')
best_lr_1_2, best_nb_1_2 = run_all_model(X_train_count_1, X_test_count_1, y_train, y_test, 'Count', 'Just Tokenization')
best_lr_1_3, best_nb_1_3 = run_all_model(X_train_tfidf_1, X_test_tfidf_1, y_train, y_test, 'TF-IDF',
                                         'Just Tokenization')

Method: Just Tokenization, Model: LogisticRegression
Best params: {'C': 1, 'max_iter': 100, 'penalty': 'l2', 'solver': 'lbfgs'}
Best accuracy: 0.8903
              precision    recall  f1-score   support

          -1       0.96      0.86      0.91       224
           0       0.84      0.97      0.90       314
           1       0.91      0.81      0.85       237

    accuracy                           0.89       775
   macro avg       0.90      0.88      0.89       775
weighted avg       0.90      0.89      0.89       775

--------------------------------------------------
Method: Just Tokenization, Model: MultinomialNB
Best params: {'alpha': 1.0}
Best accuracy: 0.8723
              precision    recall  f1-score   support

          -1       0.80      0.92      0.86       224
           0       0.95      0.89      0.92       314
           1       0.85      0.79      0.82       237

    accuracy                           0.87       775
   macro avg       0.87      0.87      0.87     

##### 2. Stremming

In [263]:
stemmer = PorterStemmer()


def stem_text(text):
    # Разбиваем на слова, применяем стемминг, объединяем обратно
    return ' '.join([stemmer.stem(word) for word in text.split()])


X_train['tweet_stemmed'] = X_train['tweet_text'].apply(stem_text)
X_test['tweet_stemmed'] = X_test['tweet_text'].apply(stem_text)

X_train_binary_2, X_test_binary_2, X_train_count_2, X_test_count_2, X_train_tfidf_2, X_test_tfidf_2 = vectorize(X_train,
                                                                                                                X_test,
                                                                                                                'tweet_stemmed')
search_all_top10(X_train_binary_2, X_train_count_2, X_train_tfidf_2)

best_lr_2_1, best_nb_2_1 = run_all_model(X_train_binary_2, X_test_binary_2, y_train, y_test, 'Binary', 'Stremming')
best_lr_2_2, best_nb_2_2 = run_all_model(X_train_count_2, X_test_count_2, y_train, y_test, 'Count', 'Stremming')
best_lr_2_3, best_nb_2_3 = run_all_model(X_train_tfidf_2, X_test_tfidf_2, y_train, y_test, 'TF-IDF', 'Stremming')

Binary shape: (3098, 4957)
Count shape: (3098, 4957)
TF-IDF shape: (3098, 4957)
Binary_vectorizer:
Score: 0.978
Tweet 1: Hi! We tried to call your number but got no response unhappy  Please share another suitable time and an alternate number for us to.. cont1
Tweet 2: Hi! We tried to call your number but got no response unhappy  Please share another suitable time and an alternate number for.. cont1
--------------------------------------------------
Score: 0.977
Tweet 1: Hi! We tried to call your number but got no response unhappy  Please share another suitable time and an alternate number.. cont1
Tweet 2: Hi! We tried to call your number but got no response unhappy  Please share another suitable time and an alternate number for.. cont1
--------------------------------------------------
Score: 0.977
Tweet 1: Hi! We tried to call your number but got no response unhappy  Please share another suitable time and an alternate number for.. cont1
Tweet 2: Hi! We tried to call your number but go

##### 3. Lemmatization

In [264]:
lemmatizer = WordNetLemmatizer()


def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN


def lemmatize_text(text):
    tokens = word_tokenize(text)
    pos_tags = pos_tag(tokens)
    lemmatized_tokens = [lemmatizer.lemmatize(word, get_wordnet_pos(pos)) for word, pos in pos_tags]
    return ' '.join(lemmatized_tokens)


X_train['tweet_lemmatized'] = X_train['tweet_text'].apply(lemmatize_text)
X_test['tweet_lemmatized'] = X_test['tweet_text'].apply(lemmatize_text)

X_train_binary_3, X_test_binary_3, X_train_count_3, X_test_count_3, X_train_tfidf_3, X_test_tfidf_3 = vectorize(X_train,
                                                                                                                X_test,
                                                                                                                'tweet_lemmatized')
search_all_top10(X_train_binary_3, X_train_count_3, X_train_tfidf_3)

best_lr_3_1, best_nb_3_1 = run_all_model(X_train_binary_3, X_test_binary_3, y_train, y_test, 'Binary', 'Lemmatization')
best_lr_3_2, best_nb_3_2 = run_all_model(X_train_count_3, X_test_count_3, y_train, y_test, 'Count', 'Lemmatization')
best_lr_3_3, best_nb_3_3 = run_all_model(X_train_tfidf_3, X_test_tfidf_3, y_train, y_test, 'TF-IDF', 'Lemmatization')

Binary shape: (3098, 5010)
Count shape: (3098, 5010)
TF-IDF shape: (3098, 5010)
Binary_vectorizer:
Score: 0.977
Tweet 1: Hi! We tried to call your number but got no response unhappy  Please share another suitable time and an alternate number.. cont1
Tweet 2: Hi! We tried to call your number but got no response unhappy  Please share another suitable time and an alternate number for us to.. cont1
--------------------------------------------------
Score: 0.977
Tweet 1: Hi! We tried to call your number but got no response unhappy  Please share another suitable time and an alternate number.. cont1
Tweet 2: Hi! We tried to call your number but got no response unhappy  Please share another suitable time and an alternate number for.. cont1
--------------------------------------------------
Score: 0.977
Tweet 1: Hi! We tried to call your number but got no response unhappy  Please share another suitable time and an alternate number.. cont1
Tweet 2: Hi Ashish! We tried to call your number but got

##### 4. Stemming + Misspellings

In [139]:
spell = SpellChecker()

# 1. Собираем уникальные слова
unique_words = set(" ".join(df['tweet_text']).split())
correction_dict = {word: spell.correction(word) or word for word in tqdm(unique_words)}


# 2. Применяем к твитам
def correct_text(text):
    return " ".join([correction_dict.get(word, word) for word in text.split()])


tqdm.pandas(desc="Processing tweets")
X_train['tweet_corrected'] = X_train['tweet_text'].progress_apply(correct_text)
X_test['tweet_corrected'] = X_test['tweet_text'].progress_apply(correct_text)

Processing tweets: 100%|██████████| 775/775 [00:00<00:00, 311120.37it/s]


In [265]:
X_train['tweet_stemmed_corrected'] = X_train['tweet_corrected'].apply(stem_text)
X_test['tweet_stemmed_corrected'] = X_test['tweet_corrected'].apply(stem_text)

X_train_binary_4, X_test_binary_4, X_train_count_4, X_test_count_4, X_train_tfidf_4, X_test_tfidf_4 = vectorize(X_train,
                                                                                                                X_test,
                                                                                                                'tweet_stemmed_corrected')
search_all_top10(X_train_binary_4, X_train_count_4, X_train_tfidf_4)

best_lr_4_1, best_nb_4_1 = run_all_model(X_train_binary_4, X_test_binary_4, y_train, y_test, 'Binary',
                                         'Stemming + Misspellings')
best_lr_4_2, best_nb_4_2 = run_all_model(X_train_count_4, X_test_count_4, y_train, y_test, 'Count',
                                         'Stemming + Misspellings')
best_lr_4_3, best_nb_4_3 = run_all_model(X_train_tfidf_4, X_test_tfidf_4, y_train, y_test, 'TF-IDF',
                                         'Stemming + Misspellings')

Binary shape: (3098, 4243)
Count shape: (3098, 4243)
TF-IDF shape: (3098, 4243)
Binary_vectorizer:
Score: 0.978
Tweet 1: Hi! We tried to call your number but got no response unhappy  Please share another suitable time and an alternate number for us to.. cont1
Tweet 2: Hi! We tried to call your number but got no response unhappy  Please share another suitable time and an alternate number for.. cont1
--------------------------------------------------
Score: 0.977
Tweet 1: Hi! We tried to call your number but got no response unhappy  Please share another suitable time and an alternate number.. cont1
Tweet 2: Hi! We tried to call your number but got no response unhappy  Please share another suitable time and an alternate number for.. cont1
--------------------------------------------------
Score: 0.977
Tweet 1: Hi! We tried to call your number but got no response unhappy  Please share another suitable time and an alternate number.. cont1
Tweet 2: Hi Ashish! We tried to call your number but

##### 5. Lemmatization + Misspellings

In [266]:
X_train['tweet_lemmatized_corrected'] = X_train['tweet_corrected'].apply(lemmatize_text)
X_test['tweet_lemmatized_corrected'] = X_test['tweet_corrected'].apply(lemmatize_text)

X_train_binary_5, X_test_binary_5, X_train_count_5, X_test_count_5, X_train_tfidf_5, X_test_tfidf_5 = vectorize(X_train,
                                                                                                                X_test,
                                                                                                                'tweet_lemmatized_corrected')
search_all_top10(X_train_binary_5, X_train_count_5, X_train_tfidf_5)

best_lr_5_1, best_nb_5_1 = run_all_model(X_train_binary_5, X_test_binary_5, y_train, y_test, 'Binary',
                                         'Lemmatization + Misspellings')
best_lr_5_2, best_nb_5_2 = run_all_model(X_train_count_5, X_test_count_5, y_train, y_test, 'Count',
                                         'Lemmatization + Misspellings')
best_lr_5_3, best_nb_5_3 = run_all_model(X_train_tfidf_5, X_test_tfidf_5, y_train, y_test, 'TF-IDF',
                                         'Lemmatization + Misspellings')

Binary shape: (3098, 4536)
Count shape: (3098, 4536)
TF-IDF shape: (3098, 4536)
Binary_vectorizer:
Score: 0.977
Tweet 1: Hi! We tried to call your number but got no response unhappy  Please share another suitable time and an alternate number.. cont1
Tweet 2: Hi! We tried to call your number but got no response unhappy  Please share another suitable time and an alternate number for us to.. cont1
--------------------------------------------------
Score: 0.977
Tweet 1: Hi! We tried to call your number but got no response unhappy  Please share another suitable time and an alternate number.. cont1
Tweet 2: Hi! We tried to call your number but got no response unhappy  Please share another suitable time and an alternate number for.. cont1
--------------------------------------------------
Score: 0.977
Tweet 1: Hi! We tried to call your number but got no response unhappy  Please share another suitable time and an alternate number for us to.. cont1
Tweet 2: Hi! We tried to call your number but 

##### 6. Lemmatization + Misspellings + Lowercasing

In [267]:
X_train['tweet_lemmatized_corrected_lower'] = X_train['tweet_corrected'].str.lower().apply(lemmatize_text)
X_test['tweet_lemmatized_corrected_lower'] = X_test['tweet_corrected'].str.lower().apply(lemmatize_text)

X_train_binary_6, X_test_binary_6, X_train_count_6, X_test_count_6, X_train_tfidf_6, X_test_tfidf_6 = vectorize(X_train,
                                                                                                                X_test,
                                                                                                                'tweet_lemmatized_corrected_lower')
search_all_top10(X_train_binary_6, X_train_count_6, X_train_tfidf_6)

best_lr_6_1, best_nb_6_1 = run_all_model(X_train_binary_6, X_test_binary_6, y_train, y_test, 'Binary',
                                         'Lemmatization + Misspellings + Lowercasing')
best_lr_6_2, best_nb_6_2 = run_all_model(X_train_count_6, X_test_count_6, y_train, y_test, 'Count',
                                         'Lemmatization + Misspellings + Lowercasing')
best_lr_6_3, best_nb_6_3 = run_all_model(X_train_tfidf_6, X_test_tfidf_6, y_train, y_test, 'TF-IDF',
                                         'Lemmatization + Misspellings + Lowercasing')

Binary shape: (3098, 4414)
Count shape: (3098, 4414)
TF-IDF shape: (3098, 4414)
Binary_vectorizer:
Score: 0.977
Tweet 1: Hi! We tried to call your number but got no response unhappy  Please share another suitable time and an alternate number.. cont1
Tweet 2: Hi! We tried to call your number but got no response unhappy  Please share another suitable time and an alternate number for us to.. cont1
--------------------------------------------------
Score: 0.977
Tweet 1: Hi! We tried to call your number but got no response unhappy  Please share another suitable time and an alternate number.. cont1
Tweet 2: Hi! We tried to call your number but got no response unhappy  Please share another suitable time and an alternate number for.. cont1
--------------------------------------------------
Score: 0.977
Tweet 1: Hi! We tried to call your number but got no response unhappy  Please share another suitable time and an alternate number for us to.. cont1
Tweet 2: Hi! We tried to call your number but 

In [268]:
print('Результаты моделей:')
results

Результаты моделей:


,Method,Vectorizer,Model,Params,Accuracy
0,Just Tokenization,Binary,LogisticRegression,"{'C': 1, 'max_iter': 100, 'penalty': 'l2', 'solver': 'lbfgs'}",0.890323
1,Just Tokenization,Binary,MultinomialNB,{'alpha': 1.0},0.872258
2,Just Tokenization,Count,LogisticRegression,"{'C': 10, 'max_iter': 100, 'penalty': 'l2', 'solver': 'lbfgs'}",0.889032
3,Just Tokenization,Count,MultinomialNB,{'alpha': 1.0},0.874839
4,Just Tokenization,TF-IDF,LogisticRegression,"{'C': 10, 'max_iter': 100, 'penalty': 'l2', 'solver': 'lbfgs'}",0.895484
5,Just Tokenization,TF-IDF,MultinomialNB,{'alpha': 1.0},0.876129
6,Stremming,Binary,LogisticRegression,"{'C': 1, 'max_iter': 100, 'penalty': 'l2', 'solver': 'lbfgs'}",0.886452
7,Stremming,Binary,MultinomialNB,{'alpha': 1.0},0.869677
8,Stremming,Count,LogisticRegression,"{'C': 1, 'max_iter': 100, 'penalty': 'l2', 'solver': 'lbfgs'}",0.892903
9,Stremming,Count,MultinomialNB,{'alpha': 1.0},0.870968


In [269]:
print('Результаты предобработки:')
X_train

Результаты предобработки:


,tweet_text,tweet_stemmed,tweet_lemmatized,tweet_corrected,tweet_stemmed_corrected,tweet_lemmatized_corrected,tweet_lemmatized_corrected_lower
0,In varsity,in varsiti,In varsity,In varsity,in varsiti,In varsity,in varsity
1,Im done trying to take men seriously happy,im done tri to take men serious happi,Im do try to take men seriously happy,i done trying to take men seriously happy,i done tri to take men serious happi,i do try to take men seriously happy,i do try to take men seriously happy
2,year old i am commenting on all the new ladylike videos. trust me. not a troll,year old i am comment on all the new ladylik videos. trust me. not a troll,year old i be comment on all the new ladylike video . trust me . not a troll,year old i am commenting on all the new ladylike videos trust me not a troll,year old i am comment on all the new ladylik video trust me not a troll,year old i be comment on all the new ladylike video trust me not a troll,year old i be comment on all the new ladylike video trust me not a troll
3,We love that our bloggers love us so much happy Thank you for such an amazing review The Perfect Glasses h,we love that our blogger love us so much happi thank you for such an amaz review the perfect glass h,We love that our blogger love u so much happy Thank you for such an amazing review The Perfect Glasses h,We love that our bloggers love us so much happy Thank you for such an amazing review The Perfect Glasses h,we love that our blogger love us so much happi thank you for such an amaz review the perfect glass h,We love that our blogger love u so much happy Thank you for such an amazing review The Perfect Glasses h,we love that our blogger love u so much happy thank you for such an amazing review the perfect glass h
4,700 personnel) of paramilitary forces in Haryana,700 personnel) of paramilitari forc in haryana,700 personnel ) of paramilitary force in Haryana,700 personnel of paramilitary forces in aryans,700 personnel of paramilitari forc in aryan,700 personnel of paramilitary force in aryan,700 personnel of paramilitary force in aryan
...,...,...,...,...,...,...,...
3093,Before going back back to jail,befor go back back to jail,Before go back back to jail,Before going back back to jail,befor go back back to jail,Before go back back to jail,before go back back to jail
3094,Aha it's okay she's going to be on Rough Night happy,aha it' okay she' go to be on rough night happi,Aha it 's okay she 's go to be on Rough Night happy,Aha it's okay she's going to be on Rough Night happy,aha it' okay she' go to be on rough night happi,Aha it 's okay she 's go to be on Rough Night happy,aha it 's okay she 's go to be on rough night happy
3095,I miss the 7/27 costumes unhappy,i miss the 7/27 costum unhappi,I miss the 7/27 costume unhappy,I miss the 7/27 costumes unhappy,i miss the 7/27 costum unhappi,I miss the 7/27 costume unhappy,i miss the 7/27 costume unhappy
3096,HAPPY BDAY MIDAWGDARNIZLLE i love you a long time happy thanks for being my best friend forever and attempting to travel on the road,happi bday midawgdarnizl i love you a long time happi thank for be my best friend forev and attempt to travel on the road,HAPPY BDAY MIDAWGDARNIZLLE i love you a long time happy thanks for be my best friend forever and attempt to travel on the road,HAPPY day MIDAWGDARNIZLLE i love you a long time happy thanks for being my best friend forever and attempting to travel on the road,happi day midawgdarnizl i love you a long time happi thank for be my best friend forev and attempt to travel on the road,HAPPY day MIDAWGDARNIZLLE i love you a long time happy thanks for be my best friend forever and attempt to travel on the road,happy day midawgdarnizlle i love you a long time happy thanks for be my best friend forever and attempt to travel on the road


##### 7. Word2Vec

In [270]:
def train_word2vec(sentences, vector_size=500, window=5, min_count=2, epochs=50):
    model = Word2Vec(
        sentences=sentences,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=4,
        epochs=epochs
    )
    return model


def vectorize_sentences(model, sentences, vector_size):
    # Преобразование предложений в векторы (среднее по словам)
    vectors = []

    for text in sentences:
        # Берем только слова, которые есть в словаре модели
        valid_words = [model.wv[word] for word in text if word in model.wv]

        if len(valid_words) > 0:
            vector = np.mean(valid_words, axis=0)
        else:
            vector = np.zeros(vector_size)

        vectors.append(vector)

    return np.array(vectors)


X_train_tokens = X_train['tweet_lemmatized_corrected'].str.split().tolist()
X_test_tokens = X_test['tweet_lemmatized_corrected'].str.split().tolist()

w2v_model = train_word2vec(X_train_tokens)

# Векторизация
X_train_w2v = vectorize_sentences(w2v_model, X_train_tokens, w2v_model.vector_size)
X_test_w2v = vectorize_sentences(w2v_model, X_test_tokens, w2v_model.vector_size)

scaler = StandardScaler()

X_train_w2v = scaler.fit_transform(X_train_w2v)
X_test_w2v = scaler.transform(X_test_w2v)

scaler = StandardScaler()

X_train_w2v = scaler.fit_transform(X_train_w2v)
X_test_w2v = scaler.transform(X_test_w2v)

clf = LogisticRegression(max_iter=500)
clf.fit(X_train_w2v, y_train)

y_pred = clf.predict(X_test_w2v)

print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.8864516129032258
